Imports and configuration

In [ ]:
# ============================================================
# BAKERY SALES & MARGIN INTELLIGENCE DASHBOARD
# STAGE 1: DATA LOADING & INITIAL INSPECTION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# 1. DEFINE PROJECT DIRECTORIES
# ============================================================

# Notebook is located inside the "notebook" folder
PROJECT_ROOT = Path.cwd().parent

# Data folder is directly inside the project root
DATA_DIR = PROJECT_ROOT / "data"

# Raw CSV files are directly inside data/
RAW_DIR = DATA_DIR


print("=" * 70)
print("PROJECT DIRECTORIES")
print("=" * 70)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data folder  : {DATA_DIR}")
print(f"Raw data     : {RAW_DIR}")


# ============================================================
# 2. VERIFY REQUIRED FILES
# ============================================================

required_files = [
    "transactions_raw.csv",
    "products.csv",
    "production_log.csv",
    "cost_changes.csv",
    "price_changes.csv",
    "holidays.csv"
]

file_status = []

for filename in required_files:

    path = RAW_DIR / filename

    file_status.append({
        "file": filename,
        "exists": path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 2)
        if path.exists() else None
    })

files_df = pd.DataFrame(file_status)

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

display(files_df)


# ============================================================
# 3. STOP IF ANY FILE IS MISSING
# ============================================================

missing_files = files_df.loc[
    ~files_df["exists"],
    "file"
].tolist()

if missing_files:

    raise FileNotFoundError(
        f"\nThe following required files are missing:\n"
        f"{missing_files}\n\n"
        f"Expected location:\n{RAW_DIR}"
    )

print("\nAll required files are present.")


# ============================================================
# 4. LOAD ALL DATASETS
# ============================================================

transactions = pd.read_csv(
    RAW_DIR / "transactions_raw.csv"
)

products = pd.read_csv(
    RAW_DIR / "products.csv"
)

production_log = pd.read_csv(
    RAW_DIR / "production_log.csv"
)

cost_changes = pd.read_csv(
    RAW_DIR / "cost_changes.csv"
)

price_changes = pd.read_csv(
    RAW_DIR / "price_changes.csv"
)

holidays = pd.read_csv(
    RAW_DIR / "holidays.csv"
)


# ============================================================
# 5. STORE DATASETS IN A DICTIONARY
# ============================================================

datasets = {
    "transactions": transactions,
    "products": products,
    "production_log": production_log,
    "cost_changes": cost_changes,
    "price_changes": price_changes,
    "holidays": holidays
}


# ============================================================
# 6. DISPLAY DATASET SHAPES
# ============================================================

print("\n" + "=" * 70)
print("DATASET SHAPES")
print("=" * 70)

for name, df in datasets.items():

    print(
        f"{name:<20} "
        f"Rows: {df.shape[0]:>8,} | "
        f"Columns: {df.shape[1]:>3}"
    )


# ============================================================
# 7. DISPLAY FIRST 5 ROWS
# ============================================================

print("\n" + "=" * 70)
print("SAMPLE DATA")
print("=" * 70)

for name, df in datasets.items():

    print(f"\n--- {name.upper()} ---")

    display(df.head())


# ============================================================
# 8. DISPLAY DATA TYPES
# ============================================================

print("\n" + "=" * 70)
print("DATA TYPES")
print("=" * 70)

for name, df in datasets.items():

    print(f"\n--- {name.upper()} ---")

    display(
        pd.DataFrame({
            "column": df.columns,
            "dtype": df.dtypes.astype(str).values
        })
    )


# ============================================================
# 9. CHECK MISSING VALUES
# ============================================================

print("\n" + "=" * 70)
print("MISSING VALUE ANALYSIS")
print("=" * 70)

missing_summary = {}

for name, df in datasets.items():

    missing_count = df.isnull().sum()

    missing_percent = (
        df.isnull().mean() * 100
    )

    summary = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_percent.round(2)
    })

    missing_summary[name] = summary

    print(f"\n--- {name.upper()} ---")

    display(
        summary[summary["missing_count"] > 0]
    )


# ============================================================
# 10. CHECK DUPLICATES
# ============================================================

print("\n" + "=" * 70)
print("DUPLICATE RECORD ANALYSIS")
print("=" * 70)

duplicate_summary = []

for name, df in datasets.items():

    duplicate_count = df.duplicated().sum()

    duplicate_summary.append({
        "dataset": name,
        "duplicate_rows": duplicate_count
    })

duplicates_df = pd.DataFrame(duplicate_summary)

display(duplicates_df)


# ============================================================
# 11. NUMERICAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("NUMERICAL SUMMARY")
print("=" * 70)

for name, df in datasets.items():

    numerical_columns = df.select_dtypes(
        include=np.number
    ).columns

    if len(numerical_columns) > 0:

        print(f"\n--- {name.upper()} ---")

        display(
            df[numerical_columns].describe().T
        )


# ============================================================
# 12. FINAL STAGE 1 SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("STAGE 1 DATA LOADING COMPLETE")
print("=" * 70)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data folder  : {DATA_DIR}")

print("\nDatasets successfully loaded:")

for name, df in datasets.items():

    print(
        f"✓ {name:<20} "
        f"{df.shape[0]:,} rows × {df.shape[1]} columns"
    )

print("\nStage 1 initial inspection completed successfully.")

In [6]:
# ============================================================
# STAGE 1 — COMPACT DATA QUALITY REPORT
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 70)
print("BAKERY SALES & MARGIN INTELLIGENCE")
print("STAGE 1 — COMPACT DATA QUALITY REPORT")
print("=" * 70)


# ------------------------------------------------------------
# 1. DATASET OVERVIEW
# ------------------------------------------------------------

overview = []

for name, df in datasets.items():

    total_missing = int(df.isna().sum().sum())
    duplicate_rows = int(df.duplicated().sum())

    # Find possible date columns
    date_columns = [
        col for col in df.columns
        if any(
            word in col.lower()
            for word in ["date", "timestamp", "time"]
        )
    ]

    date_start = None
    date_end = None

    if date_columns:

        for col in date_columns:

            dates = pd.to_datetime(
                df[col],
                errors="coerce"
            )

            if dates.notna().any():

                date_start = dates.min()
                date_end = dates.max()
                break

    overview.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_values": total_missing,
        "duplicate_rows": duplicate_rows,
        "date_start": date_start,
        "date_end": date_end
    })


overview_df = pd.DataFrame(overview)

print("\nDATASET OVERVIEW")
print("-" * 70)

display(overview_df)


# ------------------------------------------------------------
# 2. COLUMN NAMES
# ------------------------------------------------------------

print("\nCOLUMN STRUCTURE")
print("-" * 70)

for name, df in datasets.items():

    print(f"{name}:")
    print("  " + ", ".join(df.columns))


# ------------------------------------------------------------
# 3. MISSING VALUE DETAILS — ONLY IF PRESENT
# ------------------------------------------------------------

print("\nMISSING VALUES BY COLUMN")
print("-" * 70)

for name, df in datasets.items():

    missing = df.isna().sum()
    missing = missing[missing > 0]

    if len(missing) > 0:

        print(f"\n{name}:")
        
        for column, count in missing.items():

            percentage = (count / len(df)) * 100

            print(
                f"  {column}: "
                f"{count:,} "
                f"({percentage:.2f}%)"
            )

    else:

        print(f"{name}: No missing values")


# ------------------------------------------------------------
# 4. UNIQUE VALUES FOR IMPORTANT COLUMNS
# ------------------------------------------------------------

print("\nKEY CATEGORICAL VARIABLES")
print("-" * 70)

for name, df in datasets.items():

    print(f"\n{name}:")

    for column in df.columns:

        if df[column].dtype == "object":

            unique_count = df[column].nunique()

            # Only display small categorical variables
            if unique_count <= 30:

                values = (
                    df[column]
                    .dropna()
                    .unique()
                    .tolist()
                )

                print(
                    f"  {column} "
                    f"({unique_count} unique): "
                    f"{values}"
                )


# ------------------------------------------------------------
# 5. NUMERICAL RANGE CHECK
# ------------------------------------------------------------

print("\nNUMERICAL VARIABLES")
print("-" * 70)

for name, df in datasets.items():

    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns

    if len(numeric_columns) > 0:

        print(f"\n{name}:")

        for column in numeric_columns:

            print(
                f"  {column}: "
                f"min={df[column].min():.2f}, "
                f"max={df[column].max():.2f}, "
                f"mean={df[column].mean():.2f}"
            )


# ------------------------------------------------------------
# 6. SAVE COMPACT REPORT
# ------------------------------------------------------------

REPORT_DIR = PROJECT_ROOT / "reports"

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

report_file = REPORT_DIR / "stage1_data_quality_summary.csv"

overview_df.to_csv(
    report_file,
    index=False
)

print("\n" + "=" * 70)
print("STAGE 1 REPORT SAVED")
print("=" * 70)

print(f"Report: {report_file}")

print("\nStage 1 inspection completed successfully.")

BAKERY SALES & MARGIN INTELLIGENCE
STAGE 1 — COMPACT DATA QUALITY REPORT

DATASET OVERVIEW
----------------------------------------------------------------------


,dataset,rows,columns,missing_values,duplicate_rows,date_start,date_end
0,transactions,70720,8,5705,620,2026-03-01 07:02:20,2026-08-31 20:47:49
1,products,20,5,0,0,NaT,NaT
2,production_log,3680,4,343,0,2026-03-01 00:00:00,2026-08-31 00:00:00
3,cost_changes,7,3,0,0,2026-06-01 00:00:00,2026-07-01 00:00:00
4,price_changes,2,3,0,0,2026-07-01 00:00:00,2026-07-15 00:00:00
5,holidays,6,2,0,0,2026-03-20 00:00:00,2026-06-12 00:00:00



COLUMN STRUCTURE
----------------------------------------------------------------------
transactions:
  transaction_id, timestamp, product_name, quantity, unit_price, discount_pct, payment_method, cashier
products:
  product_name, category, list_price_start, unit_cost_start, is_perishable
production_log:
  date, product_name, units_produced, units_wasted_recorded
cost_changes:
  product_name, effective_date, new_unit_cost
price_changes:
  product_name, effective_date, new_list_price
holidays:
  date, holiday

MISSING VALUES BY COLUMN
----------------------------------------------------------------------

transactions:
  unit_price: 260 (0.37%)
  discount_pct: 3,350 (4.74%)
  cashier: 2,095 (2.96%)
products: No missing values

production_log:
  units_wasted_recorded: 343 (9.32%)
cost_changes: No missing values
price_changes: No missing values
holidays: No missing values

KEY CATEGORICAL VARIABLES
----------------------------------------------------------------------

transactions:
  pa